# 02 · Partición y preprocesamiento

Este notebook prepara las bases que usan todos los modelos y la aplicación:

1. Divide `train_clean.csv` en entrenamiento (70 %), validación (10 %) y prueba (20 %), y guarda `data/splits/particion.csv`.
2. Construye, **solo con el split de entrenamiento**, las reglas derivadas de los datos:
   - `data/respuestas_correctas.csv`: si cada par (pregunta, respuesta) es correcto.
   - `data/preguntas.csv`: enunciado original y normalizado de las 15 preguntas.
   - `data/misconceptions_por_pregunta.json`: ideas erróneas observadas en cada pregunta.
3. Verifica el preprocesamiento de texto definido en `src/preprocesamiento.py`.

> El `test.csv` de Kaggle solo tiene 3 filas sin etiquetas, por eso el conjunto de prueba se separa de `train_clean.csv`.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
from src import preprocesamiento as pp

pd.set_option("display.max_colwidth", 120)

## 1. Carga de datos

Se lee con `keep_default_na=False` para que el texto `"NA"` de `Misconception` no se convierta en nulo. La etiqueta objetivo es `target = Category:Misconception`.

In [2]:
df = pp.cargar_train_clean()
print(f"Filas: {len(df):,} | Etiquetas distintas (target): {df['target'].nunique()}")
df[["row_id", "QuestionId", "MC_Answer", "StudentExplanation", "target"]].head()

Filas: 36,696 | Etiquetas distintas (target): 64


,row_id,QuestionId,MC_Answer,StudentExplanation,target
0,0,31772,\( \frac{1}{3} \),0ne third is equal to tree nineth,True_Correct:NA
1,1,31772,\( \frac{1}{3} \),1 / 3 because 6 over 9 is 2 thirds and 1 third is not shaded.,True_Correct:NA
2,2,31772,\( \frac{1}{3} \),"1 3rd is half of 3 6th, so it is simplee to understand.",True_Neither:NA
3,3,31772,\( \frac{1}{3} \),1 goes into everything and 3 goes into nine,True_Neither:NA
4,4,31772,\( \frac{1}{3} \),1 out of every 3 isn't coloured,True_Correct:NA


In [3]:
conteo = df["target"].value_counts()
print("Etiquetas con menos de 10 ejemplos:", (conteo < pp.UMBRAL_ESTRATO).sum())
conteo.tail(12)

Etiquetas con menos de 10 ejemplos: 24


target
True_Misconception:Denominator-only_change                   4
True_Misconception:FlipChange                                4
True_Misconception:Definition                                3
True_Misconception:Multiplying_by_4                          3
True_Misconception:Positive                                  2
True_Misconception:Incorrect_equivalent_fraction_addition    2
True_Misconception:Subtraction                               2
True_Misconception:Not_variable                              1
True_Misconception:Longer_is_bigger                          1
True_Misconception:Whole_numbers_larger                      1
True_Misconception:Adding_across                             1
True_Misconception:Base_rate                                 1
Name: count, dtype: int64

## 2. Partición en entrenamiento, validación y prueba

- Estratificada por `target` con `random_state=42`.
- Las etiquetas con menos de 10 ejemplos se estratifican por `Category`, porque no se puede estratificar una clase con 1 o 2 filas.
- Si una etiqueta de validación o prueba no aparece en entrenamiento, esas filas pasan a entrenamiento: ningún modelo puede predecir una clase que nunca vio.

In [4]:
particion = pp.generar_particion(df)
pp.RUTA_PARTICION.parent.mkdir(parents=True, exist_ok=True)
particion.to_csv(pp.RUTA_PARTICION, index=False)

resumen = particion["split"].value_counts().rename("filas").to_frame()
resumen["proporcion"] = (resumen["filas"] / len(particion)).round(4)
resumen

,filas,proporcion
split,,
train,25688,0.7
test,7340,0.2
val,3668,0.1


In [5]:
datos = pp.aplicar_particion(df, particion)
train, val, test = datos["train"], datos["val"], datos["test"]

# La estratificación mantiene la proporción de cada categoría en los tres conjuntos
proporciones = pd.DataFrame({s: d["Category"].value_counts(normalize=True) for s, d in datos.items()}).round(4)
proporciones

,train,val,test
Category,,,
True_Correct,0.4033,0.4035,0.4034
False_Misconception,0.2577,0.2576,0.2579
False_Neither,0.1783,0.1783,0.1782
True_Neither,0.1435,0.1437,0.1435
True_Misconception,0.0111,0.0106,0.0109
False_Correct,0.0062,0.0063,0.0061


In [6]:
clases_train = set(train["target"])
print("Etiquetas en train:", len(clases_train))
print("Etiquetas de val que no están en train:", len(set(val["target"]) - clases_train))
print("Etiquetas de test que no están en train:", len(set(test["target"]) - clases_train))
print("Etiquetas presentes en test:", test["target"].nunique())

Etiquetas en train: 64
Etiquetas de val que no están en train: 0
Etiquetas de test que no están en train: 0
Etiquetas presentes en test: 51


## 3. Reglas derivadas del split de entrenamiento

Se calculan **solo con train** para no filtrar información del conjunto de prueba.

In [7]:
pp.guardar_reglas(train, df)
respuestas, preguntas, misc_por_pregunta = pp.cargar_reglas()
print(f"Pares (pregunta, respuesta): {len(respuestas)}")
respuestas.head(8)

Pares (pregunta, respuesta): 60


,QuestionId,MC_Answer,MC_Answer_norm,es_correcta,n,consistencia
0,31772,\( \frac{1}{3} \),1/3,True,1639,1.0
1,31772,\( \frac{3}{6} \),3/6,False,296,1.0
2,31772,\( \frac{3}{8} \),3/8,False,80,1.0
3,31772,\( \frac{3}{9} \),3/9,False,1388,1.0
4,31774,\( \frac{1}{12} \),1/12,True,1328,1.0
5,31774,\( \frac{1}{3} \),1/3,False,167,1.0
6,31774,\( 3 \),3,False,465,1.0
7,31774,\( \frac{6}{2} \),6/2,False,198,1.0


In [8]:
# Pares en los que la etiqueta True/False no es 100 % consistente (ruido de anotación)
respuestas[respuestas["consistencia"] < 1]

,QuestionId,MC_Answer,MC_Answer_norm,es_correcta,n,consistencia
14,31778,\( 6 \),6,True,1208,0.9959
15,31778,\( 9 \),9,False,103,0.8932


In [9]:
pd.Series({int(q): ", ".join(m) for q, m in misc_por_pregunta.items()}, name="ideas erróneas en train").to_frame()

,ideas erróneas en train
31772,"Incomplete, WNB"
31774,"FlipChange, Mult, SwapDividend"
31777,"Incomplete, Irrelevant, Wrong_fraction"
31778,"Additive, Irrelevant, WNB"
32829,"Adding_terms, Inverse_operation, Not_variable"
32833,"Duplication, Inversion, Wrong_Operation"
32835,"Ignores_zeroes, Longer_is_bigger, Shorter_is_bigger, Whole_numbers_larger"
33471,"Incomplete, Wrong_fraction"
33472,"Adding_across, Denominator-only_change, Incorrect_equivalent_fraction_addition"
33474,"Division, Subtraction"


## 4. Verificación del preprocesamiento

`preparar_dataframe` agrega las columnas que consumen los modelos:

- `explicacion_m1`: explicación normalizada y en minúsculas (M1).
- `pregunta_str`, `pregunta_respuesta`, `correcta_str`: variables categóricas (M1).
- `texto_entrada`: `Question: … Answer: … Correct: … Explanation: …` (M3).

In [10]:
ejemplo = pp.preparar_dataframe(val.sample(3, random_state=1), respuestas, preguntas)
ejemplo[["MC_Answer", "MC_Answer_norm", "es_correcta", "explicacion_m1", "texto_entrada"]]

,MC_Answer,MC_Answer_norm,es_correcta,explicacion_m1,texto_entrada
1399,\( 6 \),6,True,i did math and thats what i got,Question: A/10=9/15 What is the value of A ? Answer: 6 Correct: yes Explanation: i did math and thats what i got
1857,\( \frac{10}{15} \),10/15,False,"i think this becausee you do 5 times 2 = 10, and 3 times 5 = 15.","Question: Calculate 2/3 × 5 Answer: 10/15 Correct: no Explanation: I think this becausee you do 5 times 2 = 10, and ..."
102,\( \frac{1}{3} \),1/3,True,it actually goes in to 3/9 but it can be simplifide in to 1/3.the 3 is how many aren't shaded and the 9 is all the t...,Question: What fraction of the shape is not shaded? Give your answer in its simplest form. [Image: A triangle split ...


In [11]:
# ¿La corrección derivada coincide con el prefijo real de la etiqueta?
for nombre, d in [("val", val), ("test", test)]:
    dp = pp.preparar_dataframe(d, respuestas, preguntas)
    real = dp["Category"].str.startswith("True")
    sin_par = dp["es_correcta"].isna().sum()
    coincide = (dp["es_correcta"] == real).mean()
    print(f"{nombre}: coincidencia {coincide:.4%} | pares desconocidos: {sin_par}")

val: coincidencia 99.9455% | pares desconocidos: 0

test: coincidencia 100.0000% | pares desconocidos: 0


## 5. Máscara de etiquetas

Con la corrección de la respuesta y las ideas erróneas posibles de cada pregunta, se puede descartar de antemano gran parte de las 64 etiquetas. La tabla muestra cuántas etiquetas candidatas quedan por fila y si la etiqueta real sigue dentro de las candidatas (cota superior de lo que la máscara permite acertar).

In [12]:
import numpy as np

clases = sorted(train["target"].unique())
for nombre, d in [("val", val), ("test", test)]:
    dp = pp.preparar_dataframe(d, respuestas, preguntas)
    mascara = pp.mascara_etiquetas(clases, dp["QuestionId"], dp["es_correcta"], misc_por_pregunta)
    idx_real = dp["target"].map({c: i for i, c in enumerate(clases)}).values
    cubre = mascara[np.arange(len(dp)), idx_real].mean()
    print(f"{nombre}: candidatas por fila = {mascara.sum(1).mean():.1f} de {len(clases)} | etiqueta real dentro de la máscara: {cubre:.4%}")

val: candidatas por fila = 4.4 de 64 | etiqueta real dentro de la máscara: 99.9455%


test: candidatas por fila = 4.4 de 64 | etiqueta real dentro de la máscara: 100.0000%


## 6. Longitud de la entrada para M3

Sirve para fijar `max_length` del tokenizador de DeBERTa (se mide en palabras; los tokens suelen ser entre 1.3 y 1.6 veces más).

In [13]:
dp_train = pp.preparar_dataframe(train, respuestas, preguntas)
palabras = dp_train["texto_entrada"].str.split().str.len()
palabras.describe(percentiles=[0.5, 0.9, 0.99, 0.999]).round(1)

count    25688.0
mean        37.6
std         14.9
min         10.0
50%         36.0
90%         58.0
99%         76.0
99.9%       92.3
max        191.0
Name: texto_entrada, dtype: float64

## Archivos generados

| Archivo | Uso |
|---|---|
| `data/splits/particion.csv` | Partición común para todos los modelos |
| `data/respuestas_correctas.csv` | Corrección de cada respuesta; la app la usa para listar las 4 opciones de cada pregunta |
| `data/preguntas.csv` | Enunciados de las 15 preguntas |
| `data/misconceptions_por_pregunta.json` | Ideas erróneas posibles por pregunta (máscara de etiquetas) |